# Lab 7 · Governance: Unity Catalog privileges, row filters, column masks and ownership

Find a table's owner and type, grant and revoke read access with the three privileges a reader needs, hide rows with a row filter and an email column with a mask, then trace lineage in Catalog Explorer.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Governance and Security (15% of the exam) · **Time:** about 30 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Part 1 · A managed table and its owner

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dea_lab;
USE workspace.dea_lab;
CREATE OR REPLACE TABLE gov_customers (customer_id STRING, name STRING, email STRING, country STRING);
INSERT INTO gov_customers VALUES
  ('c-10', 'Asha',  'asha@example.com',  'UK'),
  ('c-11', 'Liam',  'liam@example.com',  'IE'),
  ('c-12', 'Chloé', 'chloe@example.com', 'FR'),
  ('c-13', 'Ravi',  'ravi@example.com',  'UK');

**Predict:** who owns this table, and is it managed or external?

In [ ]:
%sql
DESCRIBE TABLE EXTENDED gov_customers;

Scroll to **Owner** and **Type**. You created the table, so you own it, and it is **MANAGED**: Unity Catalog chose where its files live and deletes them if you drop the table. An **external** table points at a cloud path you manage, and dropping it leaves the files. (Free Edition has no external locations of your own, so this lab uses managed tables only.)

In [ ]:
%sql
SELECT table_name, table_type, table_owner
FROM workspace.information_schema.tables
WHERE table_schema = 'dea_lab'
ORDER BY table_name;

## Part 2 · The three privileges a reader needs

To read `workspace.dea_lab.gov_customers`, a person needs **USE CATALOG** on `workspace`, **USE SCHEMA** on `dea_lab` and **SELECT** on the table (or on the schema, which covers every table in it, including tables created later). `account users` is the built-in group of everyone in your account.

In [ ]:
%sql
GRANT USE SCHEMA ON SCHEMA workspace.dea_lab TO `account users`;
GRANT SELECT ON TABLE workspace.dea_lab.gov_customers TO `account users`;
SHOW GRANTS ON TABLE workspace.dea_lab.gov_customers;

In [ ]:
%sql
SHOW GRANTS ON SCHEMA workspace.dea_lab;

Open **Catalog → workspace → dea_lab → gov_customers → Permissions** to see the same grants in the UI. The Permissions tab lists grants; the **owner** is shown on the table's Overview and Details, not as a grant.

In [ ]:
%sql
REVOKE SELECT ON TABLE workspace.dea_lab.gov_customers FROM `account users`;
SHOW GRANTS ON TABLE workspace.dea_lab.gov_customers;

## Part 3 · A row filter

A row filter is a SQL function that returns true for the rows a person may see. Unity Catalog applies it to **every** query on the table.

**Predict:** you are not in a group called `dea_eu_team`. Which rows will you see?

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gov_uk_only(country STRING)
RETURN country = 'UK' OR is_account_group_member('dea_eu_team');

ALTER TABLE gov_customers SET ROW FILTER gov_uk_only ON (country);
SELECT * FROM gov_customers ORDER BY customer_id;

## Part 4 · A column mask

A column mask is a function that returns the value a person may see in that column.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION gov_mask_email(email STRING)
RETURN CASE WHEN is_account_group_member('dea_support') THEN email
            ELSE concat('***@', split_part(email, '@', 2)) END;

ALTER TABLE gov_customers ALTER COLUMN email SET MASK gov_mask_email;
SELECT * FROM gov_customers ORDER BY customer_id;

In [ ]:
%sql
DESCRIBE TABLE EXTENDED gov_customers;

Look for the row filter and column mask in the output, and on the table's page in Catalog Explorer. The data in storage is unchanged; the filter and mask apply when the table is read.

## Part 5 · Remove them

In [ ]:
%sql
ALTER TABLE gov_customers DROP ROW FILTER;
ALTER TABLE gov_customers ALTER COLUMN email DROP MASK;
SELECT count(*) AS rows_visible FROM gov_customers;

## Part 6 · Ownership and lineage

Changing the owner (shown here, not run, because giving away your table would stop you managing it in this lab):

```text
ALTER TABLE workspace.dea_lab.gov_customers OWNER TO `data-platform`;
```

Only the owner, someone with MANAGE on the object, or the owner of its schema or catalog can do this.

Now create a table from this one and look at lineage:

In [ ]:
%sql
CREATE OR REPLACE TABLE gov_customers_by_country AS
SELECT country, count(*) AS customers FROM gov_customers GROUP BY country;

Open **Catalog → workspace → dea_lab → gov_customers_by_country → Lineage** and click **See lineage graph**. Unity Catalog recorded that it was built from `gov_customers`, down to the column.

## Expected results

```text
Part 1: Owner is your user name; Type is MANAGED. information_schema lists gov_customers as MANAGED.
Part 2: SHOW GRANTS lists SELECT for account users on the table and USE SCHEMA on the schema.
        After REVOKE, the SELECT grant is gone (USE SCHEMA remains on the schema).
Part 3: only c-10 and c-13 (the UK rows).
Part 4: still the two UK rows, with emails shown as ***@example.com.
Part 5: rows_visible = 4.
Part 6: the lineage graph shows gov_customers -> gov_customers_by_country.
```

## Exam links

- SELECT alone is not enough: the reader also needs USE CATALOG and USE SCHEMA.
- Grants on a schema flow down to every table in it, including tables created later.
- Hide columns with masks (or ABAC policies on tagged columns), not by copying tables per group.
- Grant to groups, not individual users.